# PlantCLEF 2015 Siamese CNN Training

This notebook trains the two stages described in the diploma project: global `genus` S-CNN and local `species` S-CNN.

In [ ]:
%cd /content
!git clone -b robodanill/main https://github.com/robodanill/diploma.git
%cd /content/diploma
!pip install -e '.[ml]'


In [ ]:
!bash scripts/download_plantclef2015.sh data/plantclef2015/raw
!mkdir -p data/plantclef2015
!tar -xzf data/plantclef2015/raw/PlantCLEF2015TrainingData.tar.gz -C data/plantclef2015
!plant-classifier-prepare-plantclef \
  --source-root data/plantclef2015 \
  --image-root data/plantclef2015 \
  --relative-to data/plantclef2015 \
  --output data/plantclef2015/metadata.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!tar -czf data/plantclef2015/subsets/PlantCLEF2015_leaf_only.tar.gz \
  -C data/plantclef2015/subsets/leaf leaf


In [ ]:
!cp data/plantclef2015/subsets/PlantCLEF2015_leaf_only.tar.gz \
  /content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz

In [ ]:
!ls -lh /content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz

In [ ]:
# Option A: upload/copy the repository into Drive and set this path.
PROJECT_DIR = '/content/drive/MyDrive/diploma'
%cd $PROJECT_DIR

In [ ]:
!pip install -e '.[ml]'

In [ ]:
from pathlib import Path
import torch

print('CUDA:', torch.cuda.is_available())
print('Metadata exists:', Path('data/plantclef2015/metadata.csv').exists())

## Prepare normalized metadata

Run this after unpacking PlantCLEF 2015 into `data/plantclef2015`. The command scans XML annotations and keeps leaf images by default.

In [ ]:
!plant-classifier-prepare-plantclef --source-root data/plantclef2015 --output data/plantclef2015/metadata.csv

## Smoke training from Google Drive leaf-only archive

Use this path first. It trains on a small subset to verify that data loading, checkpoints, and reference index creation work before full training.

In [ ]:
!ls data/plantclef2015
!cp data/plantclef2015/leaf/metadata.csv data/plantclef2015/metadata.csv
!ls -lh data/plantclef2015/metadata.csv
!wc -l data/plantclef2015/metadata.csv

In [ ]:
%cd /content/diploma
!git pull
!pip install -e '.[ml]'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!mkdir -p data/plantclef2015
!tar -xzf /content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz -C data/plantclef2015
!cp data/plantclef2015/leaf/metadata.csv data/plantclef2015/metadata.csv
!wc -l data/plantclef2015/metadata.csv
!ls data/plantclef2015/leaf/train | head


In [ ]:
!plant-classifier-train \
  --config configs/smoke_training.yaml \
  --stage genus \
  --output checkpoints/smoke_scnn_genus_vgg16.pt


In [ ]:
!plant-classifier-train \
  --config configs/smoke_training.yaml \
  --stage species \
  --output checkpoints/smoke_scnn_species_vgg16.pt


In [ ]:
!plant-classifier-build-index \
  --config configs/smoke_training.yaml \
  --genus-checkpoint checkpoints/smoke_scnn_genus_vgg16.pt \
  --species-checkpoint checkpoints/smoke_scnn_species_vgg16.pt \
  --output checkpoints/smoke_reference_index.pt
!ls -lh checkpoints


## Sync checkpoints to Google Drive

This removes old checkpoint files from Drive unless their name contains `_best`, then copies the latest local checkpoints.

In [ ]:
!python scripts/sync_checkpoints_to_drive.py \
  --source checkpoints \
  --dest /content/drive/MyDrive/diploma_checkpoints \
  --keep-token _best
!ls -lh /content/drive/MyDrive/diploma_checkpoints


## Train S-CNN (A): genus, global view

In [ ]:
!plant-classifier-train --config configs/training.yaml --stage genus --output checkpoints/scnn_genus_vgg16.pt

## Train S-CNN (B): species, local view

In [ ]:
!plant-classifier-train --config configs/training.yaml --stage species --output checkpoints/scnn_species_vgg16.pt

## Build reference index for desktop inference

In [ ]:
!plant-classifier-build-index --config configs/training.yaml --genus-checkpoint checkpoints/scnn_genus_vgg16.pt --species-checkpoint checkpoints/scnn_species_vgg16.pt --output checkpoints/reference_index.pt